# Phase 1 - DC Offset Blocker (Design Notebook)

Tier 1 of the three-tier architecture. Runs against the Tier 2 oracle in
`python/seamic_ref/dc_offset.py`. Every number below is measured, not asserted.

- Equation: `y[n] = x[n] - x[n-1] + alpha * y[n-1]`, `alpha = exp(-2*pi*fc/fs)`
- Default: `fc = 20 Hz`, `fs = 16000 Hz` -> `alpha = 0.9921758763...`
- Suite: `python -m pytest python/tests/test_dc_offset.py` (10 tests: AC1-AC6)

## Cell 1 - Pole placement: why `exp()`, not `1 - x`

The Taylor `alpha = 1 - 2*pi*fc/fs` gives `0.9921499` vs the exact `0.9921759`.
That 2.6e-05 error shifts the 20 Hz gain by 0.03 dB - outside the +/-0.05 dB
product gate. `exp()` is exact by construction.

In [ ]:
import math
fc, fs = 20.0, 16000.0
a_exact = math.exp(-2*math.pi*fc/fs)
a_taylor = 1 - 2*math.pi*fc/fs
print(f"exact  alpha = {a_exact:.10f}")
print(f"taylor alpha = {a_taylor:.10f}  (err {a_exact-a_taylor:+.3e})")

## Cell 2 - Frequency response vs scipy (AC4)

Blue: closed-form `|H|`. Crosses: `scipy.signal.freqz`. Max err = 3.3e-16
over 400 log-spaced points.

`exp(-2*pi*fc/fs)` lands at **-2.976 dB at 20 Hz**, not -3.010 dB. The exact -3 dB
pole needs the quadratic solution, only 0.034 dB away - inaudible, not worth the
clarity cost. The spec keeps `exp()` and pins -2.976 dB +/- 0.05 in the gate.

In [ ]:
import sys; sys.path.insert(0, '../python')
import matplotlib
matplotlib.use('Agg')
import numpy as np, matplotlib.pyplot as plt
from scipy.signal import freqz
from seamic_ref import DCBlocker, analytic_magnitude
FS, A = 16000.0, DCBlocker(sample_rate_hz=16000.0).alpha
f = np.logspace(np.log10(1), np.log10(8000), 400)
_, h = freqz([1,-1],[1,-A], worN=2*np.pi*f/FS)
plt.semilogx(f, 20*np.log10(analytic_magnitude(f,A,FS)), label='closed form')
plt.semilogx(f[::20], 20*np.log10(np.abs(h[::20])), 'x', label='scipy freqz')
plt.axvline(20, ls='--', label='20 Hz'); plt.legend(); plt.grid(True, which='both')
plt.xlabel('Hz'); plt.ylabel('dB'); plt.title('DC blocker magnitude'); plt.close()
m20 = analytic_magnitude(np.array([20.0]),A,FS)[0]
print(f"gain @20Hz = {20*np.log10(m20):.3f} dB")
print(f"max err vs freqz = {np.max(np.abs(analytic_magnitude(f,A,FS)-np.abs(h))):.1e}")

## Cell 3 - Step response vs `lfilter` (AC1)

Zero initial state (`x_prev = y_prev = 0`) matches the canonical filter: max err
= 0.0 on the 64-sample step (<= 7e-15 on all probed inputs). First sample:
`y[0] = x[0]` - unity gain, no click, so no
priming or fade-in is needed.

In [ ]:
from scipy.signal import lfilter
xs = np.ones(64)
blk = DCBlocker(sample_rate_hz=FS); blk.alpha = A
mine = blk.process_block(xs); ref = lfilter([1,-1],[1,-A],xs)
print("first 8:", np.round(mine[:8],6))
print(f"max err vs lfilter: {np.max(np.abs(mine-ref)):.2e} | y[0]={mine[0]:.4f} (unity gain)")

## Cell 4 - DC rejection and settling (AC2)

A 0.5 DC step decays as `0.5*alpha^n`. After 8000 samples (0.5 s) the residue is
below 1e-28 - the tail is monotone, never grows. Settling to +/-1e-4 of full scale
after a +1 step: **1173 samples (~73.3 ms at 16 kHz)**.

In [ ]:
blk = DCBlocker(sample_rate_hz=FS); blk.set_bypass(False)
out = blk.process_block(np.full(8000, 0.5))
tail = np.abs(out[100:])
print(f"final sample {out[-1]:.3e}; monotone: {bool(np.all(np.diff(tail) <= 1e-15))}")
b2 = DCBlocker(sample_rate_hz=FS); b2.set_bypass(False)
o2 = np.abs(b2.process_block(np.ones(8000)))
n = next(i for i,v in enumerate(o2) if v < 1e-4)
print(f"settle to 1e-4 after +1 step: {n} samples (~{1000*n/FS:.1f} ms)")

## Cell 5 - FORBIDDEN: clamping `y` to [-1, 1] (AC5)

A full-scale downward step (+1 to -1) drives the true output to ~-1.98. Clamping
`y_prev` corrupts the state (peak error ~0.98) and the error persists ~584
samples (~36 ms). The
blocker must let the output exceed the rails - headroom is downstream's job.

In [ ]:
xs = np.concatenate([np.ones(500), -np.ones(5000)])
gblk = DCBlocker(sample_rate_hz=FS); gblk.set_bypass(False)
good = gblk.process_block(xs)
xp = yp = 0.0; bad = np.empty_like(xs)
for n,x in enumerate(xs):
    y = x - xp + A*yp
    y = max(-1.0, min(1.0, y))
    xp, yp = x, y; bad[n] = y
d = np.abs(good-bad); k = int(np.argmax(d > 0.01)); j = len(d)-int(np.argmax(d[::-1] > 0.01))
print(f"peak |good-bad| = {d.max():.3f} (true output reaches {good.min():.3f})")
print(f"corrupted for ~{j-k} samples (~{1000*(j-k)/FS:.0f} ms)")

## Cell 6 - State precision: float32 vs float64 (AC6)

A 0.5 DC pulse of 100 samples, then silence. The pole residue `alpha^n*C` falls
below the float32 LSB grid (~1e-7) long before float64 stops resolving it:
rms32 ~= 8.8e-44 vs rms64 ~= 2.0e-138 in the late tail (ratio ~= 2.1e+189).
**Product state must be
float64** (or wider than the audio word). Informational, not a product gate.

In [ ]:
f32 = np.float32; a32 = f32(A)
xs = np.zeros(48000); xs[:100] = 0.5
e2 = DCBlocker(sample_rate_hz=FS); e2.set_bypass(False)
e64 = np.mean(e2.process_block(xs)[40000:]**2)
xp = yp = f32(0.0); y32 = np.empty_like(xs)
for k,x in enumerate(xs):
    y = f32(f32(x) - xp + a32*yp); xp, yp = f32(x), y; y32[k] = float(y)
e32 = np.mean(y32[40000:]**2)
print(f"tail rms: float32 {np.sqrt(e32):.2e} vs float64 {np.sqrt(e64):.2e} (ratio {e32/e64:.2e})")